# 18 — Fine-tuning a pretrained DoMINO

Notebook 10 and the DoMINO deployment pages cover running a *pretrained* checkpoint
in the loop. This notebook covers the step after that: adapting one to your own
Kratos data without training a 10 M-parameter model from scratch.

`domino_finetune` ships the two recipes described in [Point
Clouds](../../../../docs/pages/Applications/PhysicsNeMo_Application/Point_Clouds/Point_Clouds.md):

1. **Predictor-corrector** — `Y_finetuned = Y_predictor + Y_corrector`, the frozen
   checkpoint plus a small head trained on its error.
2. **LoRA** — low-rank adapters on the pretrained weights, merged back into an
   ordinary `.mdlus`.

Everything here runs on a synthetic stand-in predictor and a tiny synthetic DoMINO,
so the notebook needs neither the network nor the 48 MB pretrained checkpoint.

In [ ]:
from pathlib import Path

import numpy
import torch
import matplotlib.pyplot as plt

import KratosMultiphysics as Kratos
Kratos.Logger.GetDefaultOutput().SetSeverity(Kratos.Logger.Severity.WARNING)
from KratosMultiphysics.PhysicsNeMoApplication.training import domino_finetune

torch.manual_seed(0)
numpy.random.seed(0)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

# a strip of surface entities standing in for a DoMINO surface mesh
model = Kratos.Model()
model_part = model.CreateModelPart("Surface")
model_part.AddNodalSolutionStepVariable(Kratos.PRESSURE)
for i in range(400):
    x = i / 399.0
    model_part.CreateNewNode(i + 1, x, 0.15 * numpy.sin(6.0 * x), 0.0)

coordinates = numpy.array([[n.X, n.Y, n.Z] for n in model_part.Nodes])

# ground truth we want the fine-tuned model to reproduce
truth = (0.8 * numpy.sin(3.0 * coordinates[:, 0])
         + 0.4 * coordinates[:, 1]
         - 0.2 * numpy.cos(5.0 * coordinates[:, 0]))[:, None]
print("entities:", len(coordinates), " truth range:",
      f"{truth.min():+.3f} .. {truth.max():+.3f}")

## The frozen predictor

Stand-in for the pretrained checkpoint. What matters for the recipe is only that it
is **frozen** and **wrong in a structured way** — a real pretrained DoMINO is wrong
about your geometry in exactly this sense, not randomly.

`CacheBasePredictions` is the function used against a real checkpoint: it runs the
predictor once per case, so the predictor never appears inside the training loop and
the corrector's cost is independent of the predictor's size. Here the stand-in is
cheap enough to call directly, so we cache its output the same way by hand.

In [ ]:
predictor = torch.nn.Sequential(
    torch.nn.Linear(3, 32), torch.nn.Tanh(),
    torch.nn.Linear(32, 1)).double()
for parameter in predictor.parameters():
    parameter.requires_grad_(False)          # frozen, like the pretrained checkpoint
predictor.eval()

with torch.no_grad():
    base_prediction = predictor(torch.from_numpy(coordinates)).numpy()

base_error = float(numpy.abs(base_prediction - truth).mean())
print(f"mean abs error of the frozen predictor: {base_error:.4f}")

## An untrained corrector is exactly the predictor

`CreateCorrector` zero-initializes its last layer on purpose. Fine-tuning therefore
starts from the pretrained model's own answer and can only improve on it — there is
no initial transient in which the combined model is worse than the checkpoint you
started from.

In [ ]:
features = numpy.hstack([coordinates, base_prediction])

corrector = domino_finetune.CreateCorrector(
    features.shape[1], truth.shape[1], hidden=32, n_layers=3).double()

untrained = domino_finetune.ApplyCorrector(corrector, base_prediction, features)
print("max |untrained - predictor| =", numpy.abs(untrained - base_prediction).max())
assert numpy.abs(untrained - base_prediction).max() < 1e-12, (
    "an untrained corrector must be exactly the identity on the predictor")

## Training the corrector on the residual

The corrector is fitted on `ground_truth - base_prediction`. Both sides live in the
predictor's own output space — with a real checkpoint that space is *normalized*,
which is why `CacheBasePredictions` returns raw normalized output rather than
physical values. Forming the residual in one space consistently is the whole
requirement.

In [ ]:
residuals = truth - base_prediction

history = domino_finetune.TrainCorrector(
    corrector, features, residuals, epochs=400, learning_rate=5e-3)

combined = domino_finetune.ApplyCorrector(corrector, base_prediction, features)
tuned_error = float(numpy.abs(combined - truth).mean())
print(f"mean abs error  frozen predictor: {base_error:.4f}"
      f"   ->  predictor + corrector: {tuned_error:.4f}")
assert tuned_error < 0.5 * base_error, (
    f"fine-tuning did not improve on the predictor: {base_error:.4f} -> {tuned_error:.4f}")

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(history)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE"); axes[0].set_title("Corrector training loss")
order = numpy.argsort(coordinates[:, 0])
axes[1].plot(coordinates[order, 0], truth[order], "k-", label="ground truth")
axes[1].plot(coordinates[order, 0], base_prediction[order], "r--", label="frozen predictor")
axes[1].plot(coordinates[order, 0], combined[order], "b-.", label="predictor + corrector")
axes[1].set_xlabel("x"); axes[1].legend(); axes[1].set_title("Fine-tuned surface field")
plt.tight_layout(); plt.savefig("output/finetune.png", dpi=120); plt.show()

# writing the fine-tuned field back onto the model part, as a deployment would
for node, value in zip(model_part.Nodes, combined[:, 0]):
    node.SetSolutionStepValue(Kratos.PRESSURE, float(value))

## LoRA on the pretrained weights

The other recipe adapts the checkpoint itself. `ApplyLora` wraps its linear layers in
low-rank adapters and freezes everything else, so only a small fraction of the
parameters is trainable; `MergeAndSave` folds the adapters back in and writes an
ordinary `.mdlus`.

A tiny synthetic DoMINO stands in for the pretrained one here — the same reduced
configuration the tests use, so this cell needs no checkpoint.

In [ ]:
import copy
from physicsnemo.models.domino import DoMINO
from physicsnemo.models.domino.config import DEFAULT_MODEL_PARAMS

parameters = copy.deepcopy(DEFAULT_MODEL_PARAMS)
parameters["model_type"] = "surface"
parameters["interp_res"] = [8, 8, 8]
parameters["num_neighbors_surface"] = 4
parameters["geometry_rep"]["base_filters"] = 4
parameters["geometry_rep"]["geo_conv"]["base_neurons"] = 8
parameters["geometry_rep"]["geo_conv"]["surface_neighbors_in_radius"] = [4, 4, 4]
parameters["geometry_rep"]["geo_conv"]["volume_neighbors_in_radius"] = [4, 4, 4, 4]
parameters["geometry_rep"]["geo_processor"]["base_filters"] = 4
parameters["geometry_local"]["base_layer"] = 8
parameters["geometry_local"]["surface_neighbors_in_radius"] = [4, 4]
parameters["geometry_local"]["volume_neighbors_in_radius"] = [4, 4]
parameters["nn_basis_functions"]["base_layer"] = 8
parameters["aggregation_model"]["base_layer"] = 8
parameters["position_encoder"]["base_neurons"] = 8

pretrained = DoMINO(input_features=3, output_features_vol=None,
                    output_features_surf=1, global_features=2,
                    model_parameters=parameters)
total = sum(p.numel() for p in pretrained.parameters())

adapted, wrapped, trainable = domino_finetune.ApplyLora(pretrained, rank=4)
print(f"layers wrapped: {wrapped}   trainable: {trainable:,} / {total:,} "
      f"({100.0 * trainable / total:.1f} %)")
assert 0 < trainable < total, (
    f"LoRA must leave only a fraction trainable: {trainable} of {total}")

domino_finetune.MergeAndSave(adapted, "output/finetuned.mdlus")
print("merged checkpoint:", OUTPUT.joinpath("finetuned.mdlus").stat().st_size, "bytes")

## Where this leaves you

The merged `.mdlus` is an ordinary checkpoint: `model_registry` loads it and
`DominoInferenceProcess` deploys it with no change to the **model** settings.

It is *not* free of the de-normalization requirement. A fine-tuned model still lives
in the pretrained model's normalized output space, so
`scaling_factors_file`/`normalization`/`redimensionalize` remain as necessary as they
were for the checkpoint it was adapted from — writing raw output onto Kratos entities
is wrong by roughly three orders of magnitude.

Accuracy claims for either recipe belong to NVIDIA, who describe their own fine-tuning
results as preliminary and report them on 18 training samples. Nothing here reproduces
or endorses a number.